In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import os

PLOT_DIR = '../plots/animals_in_trading/'
TOKENS = ['USDT', 'USDC', 'DAI', 'UST', 'PAX']

DEPEG_DATE = pd.Timestamp('2022-05-09')
DATE_START = pd.Timestamp('2022-05-02')
DATE_END = pd.Timestamp('2022-05-20')

SIZE_LABELS = {
    'size_1m_plus': '$1M+',
    'size_100k_1m': '$100K-$1M',
    'size_1k_100k': '$1K-$100K',
    'size_under_1k': '<$1K'
}

SIZE_CATEGORIES = ['size_1m_plus', 'size_100k_1m', 'size_1k_100k', 'size_under_1k']
WHALE_THRESHOLD = 1_000_000


def load_data():
    """Load and merge transfer data with price data."""
    DAI_price_df = pd.read_csv("../data/DAI/DAI_full_dataset.csv")
    PAX_price_df = pd.read_csv("../data/PAX/PAX_full_dataset.csv")
    USDC_price_df = pd.read_csv("../data/USDC/USDC_full_dataset.csv")
    USDT_price_df = pd.read_csv("../data/USDT/USDT_full_dataset.csv")
    UST_price_df = pd.read_csv("../data/UST/UST_full_dataset.csv")

    df = pd.read_csv("../../data/token_transfers_cleaned.csv")
    df['date'] = pd.to_datetime(df['date']).dt.date

    DAI_price_df['token_name'] = 'DAI'
    PAX_price_df['token_name'] = 'PAX'
    USDC_price_df['token_name'] = 'USDC'
    USDT_price_df['token_name'] = 'USDT'
    UST_price_df['token_name'] = 'UST'

    price_df = pd.concat([DAI_price_df, PAX_price_df, USDC_price_df, 
                          USDT_price_df, UST_price_df], ignore_index=True)
    price_df['date'] = pd.to_datetime(price_df['date']).dt.date

    df = df.merge(price_df[['date', 'token_name', 'close']], 
                  on=['date', 'token_name'], how='left')
    df['value_usd'] = df['value'] * df['close']
    
    df['date_ts'] = pd.to_datetime(df['date'])
    df = df[(df['date_ts'] >= DATE_START) & (df['date_ts'] <= DATE_END)]
    df = df.drop(columns=['date_ts'])
    
    return df, price_df


def add_depeg_line(ax, alpha=0.7, linewidth=1.5):
    """Add vertical line for the depeg date."""
    ax.axvline(DEPEG_DATE, color='red', linestyle='--', 
               linewidth=linewidth, alpha=alpha)


def classify_transaction_size(v):
    """Classify individual transaction by size."""
    if v >= 1_000_000:
        return 'size_1m_plus'
    if v >= 100_000:
        return 'size_100k_1m'
    if v >= 1_000:
        return 'size_1k_100k'
    return 'size_under_1k'


def classify_address_size(daily_volume):
    """Classify address by daily volume."""
    if daily_volume >= WHALE_THRESHOLD:
        return 'whale'
    return 'other'


def add_classifications(df, price_df):
    """Add size and performance based classifications."""
    df['date_ts'] = pd.to_datetime(df['date'])
    
    df['size_category'] = df['value_usd'].apply(classify_transaction_size)
    
    daily_addr_vol = df.groupby(['date_ts', 'token_name', 'from_address'])['value_usd'].sum().reset_index()
    daily_addr_vol.columns = ['date_ts', 'token_name', 'from_address', 'daily_volume']
    daily_addr_vol['addr_size_category'] = daily_addr_vol['daily_volume'].apply(classify_address_size)
    
    df = df.merge(
        daily_addr_vol[['date_ts', 'token_name', 'from_address', 'addr_size_category']], 
        on=['date_ts', 'token_name', 'from_address'], 
        how='left'
    )
    
    price_df = price_df.copy()
    price_df['date'] = pd.to_datetime(price_df['date']).dt.date
    price_df_sorted = price_df.sort_values(['token_name', 'date'])
    price_df_sorted['future_close'] = price_df_sorted.groupby('token_name')['close'].shift(-1)
    future_prices = price_df_sorted[['date', 'token_name', 'future_close']].copy()

    df = df.merge(future_prices, on=['date', 'token_name'], how='left')
    
    df['markout_1d'] = df['value_usd'] * (df['close'] - df['future_close'])

    address_markouts = df.groupby('from_address')['markout_1d'].sum().reset_index()
    address_markouts.columns = ['from_address', 'cumulative_markout']
    threshold = address_markouts['cumulative_markout'].quantile(0.99)
    address_markouts['is_shark'] = address_markouts['cumulative_markout'] >= threshold

    df = df.merge(address_markouts[['from_address', 'is_shark']], 
                  on='from_address', how='left')
    df['is_shark'] = df['is_shark'].fillna(False)
    df = df.drop(columns=['future_close'], errors='ignore')
    
    return df


def plot_whale_volume_proportion(df):
    """Daily volume and proportion for whale addresses ($1M+/day)."""
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))

    for idx, token in enumerate(TOKENS):
        ax = axes[idx]
        data = df[df['token_name'] == token]

        daily_total = data.groupby('date_ts')['value_usd'].sum()
        daily_whale = data[data['addr_size_category'] == 'whale'].groupby('date_ts')['value_usd'].sum()
        daily_whale = daily_whale.reindex(daily_total.index, fill_value=0)
        pct_whale = (daily_whale / daily_total * 100).fillna(0)

        ax.bar(daily_whale.index, daily_whale.values / 1e9,
               width=0.8, alpha=0.7, color='#1a5276', label='Whale Volume')

        ax2 = ax.twinx()
        ax2.plot(pct_whale.index, pct_whale.values, color='#c0392b',
                 linewidth=2, marker='o', markersize=4, label='Whale %')

        add_depeg_line(ax)

        ax.set_title(f'{token}', fontsize=16, fontweight='bold')
        ax.set_ylabel('Volume (Billions USD)', fontsize=13)
        ax2.set_ylabel('Whale %', fontsize=13, color='#c0392b')
        ax2.tick_params(axis='y', labelcolor='#c0392b')
        ax2.set_ylim(0, 100)

        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=12)

        if idx == 0:
            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)

    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/animal_plot1_whale_volume_proportion.png', dpi=150, bbox_inches='tight')
    plt.close()


def plot_stacked_size_composition(df):
    """Daily stacked area showing volume % by transaction size."""
    colors = ['#1a5276', '#2980b9', '#5dade2', '#aed6f1']

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))

    for idx, token in enumerate(TOKENS):
        ax = axes[idx]
        data = df[df['token_name'] == token]

        daily = data.groupby(['date_ts', 'size_category'])['value_usd'].sum().unstack(fill_value=0)
        for cat in SIZE_CATEGORIES:
            if cat not in daily.columns:
                daily[cat] = 0
        daily = daily[SIZE_CATEGORIES]
        daily_pct = daily.div(daily.sum(axis=1), axis=0) * 100

        ax.stackplot(daily_pct.index, [daily_pct[c] for c in SIZE_CATEGORIES],
                     labels=[SIZE_LABELS[c] for c in SIZE_CATEGORIES],
                     colors=colors, alpha=0.85)

        top_mean = daily_pct['size_1m_plus'].mean()
        if top_mean > 5 and len(daily_pct) > 0:
            mid_idx = len(daily_pct) // 2
            mid_date = daily_pct.index[mid_idx]
            top_val = daily_pct['size_1m_plus'].iloc[mid_idx]
            if top_val > 10:
                ax.annotate('$1M+', xy=(mid_date, 100 - top_val / 2),
                           fontsize=12, ha='center', va='center',
                           color='white', fontweight='bold')

        add_depeg_line(ax)

        ax.set_title(f'{token}', fontsize=16, fontweight='bold')
        ax.set_ylim(0, 100)
        ax.set_ylabel('Volume %', fontsize=13)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=12)

        if idx == 0:
            ax.legend(loc='lower left', fontsize=11, title='Size Range')

    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/animal_plot2_stacked_size_composition.png', dpi=150, bbox_inches='tight')
    plt.close()


def plot_shark_volume_proportion(df):
    """Daily volume and proportion for sharks (top 1% by markout)."""
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))

    for idx, token in enumerate(TOKENS):
        ax = axes[idx]
        data = df[df['token_name'] == token]

        daily_total = data.groupby('date_ts')['value_usd'].sum()
        daily_shark = data[data['is_shark'] == True].groupby('date_ts')['value_usd'].sum()
        daily_shark = daily_shark.reindex(daily_total.index, fill_value=0)
        pct_shark = (daily_shark / daily_total * 100).fillna(0)

        ax.bar(daily_shark.index, daily_shark.values / 1e9,
               width=0.8, alpha=0.7, color='#27ae60', label='Shark Volume')

        ax2 = ax.twinx()
        ax2.plot(pct_shark.index, pct_shark.values, color='#8e44ad',
                 linewidth=2, marker='o', markersize=4, label='Shark %')

        add_depeg_line(ax)

        ax.set_title(f'{token}', fontsize=16, fontweight='bold')
        ax.set_ylabel('Volume (Billions USD)', fontsize=13)
        ax2.set_ylabel('Shark %', fontsize=13, color='#8e44ad')
        ax2.tick_params(axis='y', labelcolor='#8e44ad')

        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=12)

        if idx == 0:
            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)

    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/animal_plot3_shark_markout.png', dpi=150, bbox_inches='tight')
    plt.close()


def run_all():
    os.makedirs(PLOT_DIR, exist_ok=True)
    print(f"Date range: {DATE_START.date()} to {DATE_END.date()}")

    df, price_df = load_data()
    df = add_classifications(df, price_df)

    for i, plot_fn in enumerate([plot_whale_volume_proportion, plot_stacked_size_composition, plot_shark_volume_proportion], 1):
        print(f"Generating Plot {i}: {plot_fn.__doc__}")
        plot_fn(df)

    print(f"\nSaved all plots to {PLOT_DIR}/")
    return df


if __name__ == "__main__":
    df = run_all()

Date range: 2022-05-02 to 2022-05-20
Generating Plot 1: Daily volume and proportion for whale addresses ($1M+/day).
Generating Plot 2: Daily stacked area showing volume % by transaction size.
Generating Plot 3: Daily volume and proportion for sharks (top 1% by markout).

Saved all plots to ../plots/animals_in_trading//
